# Lane 7: CWS-FM - Google Colab Runner
**Project:** CWS-FM (Causal Wavefront Simplex Flow Matching)  
**Author:** Jaime Jarauta  
**Objective:** Non-Autoregressive language generation via Riemannian optimal transport on the Bhattacharyya sphere in 4 to 8 parallel ODE steps.

**Hardware Target:** Standard T4 GPU | **Max Runtime:** <= 15 minutes | **Cloud Credit Discipline:** Auto-Unassign On Completion or Error

In [ ]:
# Step 1: Credit Guardrail & Auto-Kill On Error
import sys, os, time, math
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython import get_ipython

# Safety Fallback: Automatically unassign Colab VM on ANY error to prevent idle burn
def sync_all_wandb_runs():
    """Flushes all offline or pending W&B runs to cloud before VM destruction."""
    import glob, subprocess
    candidates = glob.glob("/tmp/wandb/**/offline-run-*", recursive=True) + glob.glob("**/offline-run-*", recursive=True)
    if candidates:
        print(f"\n[W&B Guard] Detected {len(candidates)} offline run(s). Syncing to cloud before teardown...")
        for p in set(candidates):
            try:
                print(f"[W&B Guard] Syncing {p}...")
                subprocess.run(["wandb", "sync", p], check=False)
            except Exception as e:
                print(f"[W&B Warning] Sync failed for {p}: {e}")
    else:
        try:
            subprocess.run(["wandb", "sync", "--include-offline", "/tmp/wandb"], check=False)
            subprocess.run(["wandb", "sync", "--include-offline", "wandb"], check=False)
        except Exception:
            pass

def auto_kill_on_error(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("\n[Credit Guard Alert] Unhandled error caught! Flushing telemetry and unassigning Colab GPU...")
    try:
        sync_all_wandb_runs()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), auto_kill_on_error)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# Step 1b: Clone Patched Repository & Install Dependencies
import os, sys, subprocess
if 'google.colab' in sys.modules:
    repo_dir = 'cws-fm'
    if not os.path.exists(repo_dir):
        print('Cloning cws-fm (branch: feature/initial-implementation)...')
        subprocess.run(['git', 'clone', '-b', 'feature/initial-implementation', 'https://github.com/jaimejarauta/cws-fm.git'], check=True)
    if os.path.exists(repo_dir):
        os.chdir(repo_dir)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', 'wandb', 'pytest'], check=True)
    if 'src' not in sys.path:
        sys.path.insert(0, 'src')
    print('Repository ready:', os.getcwd())


In [ ]:
# Step 1c: Secure Weights & Biases Authentication
# Securely reads from Colab Secrets (Key icon on left sidebar) or reuses ~/.netrc.
import os
import wandb

# 1. Attempt reading from Google Colab Secrets across candidate names
try:
    from google.colab import userdata
    for secret_name in ['WANDB_API_KEY', 'wandb_api_key', 'WANDB', 'WAN', 'wandb']:
        try:
            val = userdata.get(secret_name)
            if val and len(val.strip()) > 10:
                os.environ['WANDB_API_KEY'] = val.strip()
                print(f"[W&B] Loaded API key from Colab Secrets ('{secret_name}').")
                break
        except Exception:
            pass
except Exception:
    pass

# 2. Fallback: check ~/.netrc if previously logged in
netrc_path = os.path.expanduser('~/.netrc')
if 'WANDB_API_KEY' not in os.environ and os.path.exists(netrc_path):
    try:
        with open(netrc_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2 and parts[0] == 'password' and len(parts[1]) > 10:
                    os.environ['WANDB_API_KEY'] = parts[1]
                    print("[W&B] Restored WANDB_API_KEY automatically from ~/.netrc.")
                    break
    except Exception:
        pass

# 3. Login to W&B without prompting if credentials exist
if 'WANDB_API_KEY' in os.environ:
    wandb.login(key=os.environ['WANDB_API_KEY'], relogin=False)
else:
    print("[W&B] Note: To avoid interactive prompt, name your secret 'WANDB_API_KEY' and paste full key into Value.")
    wandb.login(relogin=False)

os.environ['WANDB_ENTITY'] = 'research-projekts'
print('[W&B] Authenticated successfully. Target entity: research-projekts')


In [ ]:
# Step 2: Self-Contained Riemannian Simplex Flow Matching Engine
def bhattacharyya_sphere_map(p: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    p_clamped = torch.clamp(p, min=eps)
    p_norm = p_clamped / torch.sum(p_clamped, dim=-1, keepdim=True)
    return torch.sqrt(p_norm)

def inverse_bhattacharyya_sphere_map(u: torch.Tensor) -> torch.Tensor:
    p = torch.square(torch.clamp(u, min=0.0))
    return p / torch.clamp(torch.sum(p, dim=-1, keepdim=True), min=1e-12)

def wavefront_phase(t: torch.Tensor, seq_len: int, alpha: float = 0.50) -> torch.Tensor:
    device = t.device
    pos = torch.linspace(0.0, 1.0, seq_len, device=device).unsqueeze(0)
    denom = max(1e-5, 1.0 - alpha)
    tau = (t - alpha * pos) / denom
    return torch.clamp(tau, 0.0, 1.0)

def geodesic_sphere_path(u0: torch.Tensor, u1: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
    inner = torch.sum(u0 * u1, dim=-1, keepdim=True).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
    theta = torch.acos(inner)
    sin_theta = torch.sin(theta).clamp_min(1e-5)
    c0 = torch.sin((1.0 - tau) * theta) / sin_theta
    c1 = torch.sin(tau * theta) / sin_theta
    gamma = c0 * u0 + c1 * u1
    return gamma / torch.norm(gamma, p=2, dim=-1, keepdim=True).clamp_min(1e-12)

def geodesic_sphere_velocity(u0: torch.Tensor, u1: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
    inner = torch.sum(u0 * u1, dim=-1, keepdim=True).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
    theta = torch.acos(inner)
    sin_theta = torch.sin(theta).clamp_min(1e-5)
    v0 = (-theta * torch.cos((1.0 - tau) * theta)) / sin_theta
    v1 = (theta * torch.cos(tau * theta)) / sin_theta
    vel = v0 * u0 + v1 * u1
    gamma = geodesic_sphere_path(u0, u1, tau)
    return vel - torch.sum(gamma * vel, dim=-1, keepdim=True) * gamma

def riemannian_exp_map(u: torch.Tensor, v: torch.Tensor, step_size: float) -> torch.Tensor:
    delta = v * step_size
    norm_delta = torch.norm(delta, p=2, dim=-1, keepdim=True).clamp_min(1e-8)
    out = torch.cos(norm_delta) * u + torch.sin(norm_delta) * (delta / norm_delta)
    return out / torch.norm(out, p=2, dim=-1, keepdim=True).clamp_min(1e-12)

print("Bhattacharyya Fisher-Rao geometry engine initialized.")

In [ ]:
# Step 3: Neural Velocity Field & Riemannian Heun ODE Sampler
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        self.mlp = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.SiLU(), nn.Linear(d_model * 2, d_model))
    def forward(self, t: torch.Tensor) -> torch.Tensor:
        if t.dim() == 1: t = t.unsqueeze(-1)
        half_dim = self.d_model // 2
        emb_scale = math.log(10000.0) / (half_dim - 1)
        freqs = torch.exp(torch.arange(half_dim, device=t.device) * -emb_scale)
        args = t * freqs.unsqueeze(0)
        return self.mlp(torch.cat([torch.sin(args), torch.cos(args)], dim=-1))

class SimplexVelocityField(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 128, num_layers: int = 4, num_heads: int = 4, max_seq_len: int = 128):
        super().__init__()
        self.in_proj = nn.Linear(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.time_emb = SinusoidalTimeEmbedding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=num_heads, dim_feedforward=4*d_model, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.out_proj = nn.Linear(d_model, vocab_size, bias=False)
    def forward(self, u_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        B, L, V = u_t.shape
        if not isinstance(t, torch.Tensor): t = torch.tensor([[t]], device=u_t.device, dtype=u_t.dtype)
        if t.dim() == 1: t = t.unsqueeze(-1)
        if t.size(0) != B: t = t.expand(B, 1)
        pos = torch.arange(L, device=u_t.device).unsqueeze(0)
        h = self.in_proj(u_t) + self.pos_emb(pos) + self.time_emb(t).unsqueeze(1)
        out = self.transformer(h)
        v_raw = self.out_proj(out)
        # Orthogonal tangent projection
        return v_raw - torch.sum(u_t * v_raw, dim=-1, keepdim=True) * u_t

def sample_cws_fm(model, seq_len: int, vocab_size: int, num_steps: int = 8, alpha: float = 0.50, confidence_lock_threshold: float = 0.85, device="cuda"):
    model.eval()
    h = 1.0 / num_steps
    u_curr = torch.ones(1, seq_len, vocab_size, device=device) / math.sqrt(vocab_size)
    locked_mask = torch.zeros(1, seq_len, dtype=torch.bool, device=device)
    locked_tokens = torch.zeros(1, seq_len, dtype=torch.long, device=device)
    with torch.no_grad():
        for step in range(num_steps):
            t_curr = torch.tensor([[step * h]], device=device)
            t_next = torch.tensor([[(step + 1) * h]], device=device)
            tau_next = wavefront_phase(t_next, seq_len, alpha=alpha).unsqueeze(-1)
            v_curr = model(u_curr, t_curr)
            u_euler = torch.clamp(riemannian_exp_map(u_curr, v_curr, h), min=0.0)
            u_euler = u_euler / torch.norm(u_euler, p=2, dim=-1, keepdim=True)
            v_next = model(u_euler, t_next)
            u_next = torch.clamp(riemannian_exp_map(u_curr, 0.5 * (v_curr + v_next), h), min=0.0)
            u_next = u_next / torch.norm(u_next, p=2, dim=-1, keepdim=True)
            # Adaptive confidence & terminal wavefront locking
            p_next = inverse_bhattacharyya_sphere_map(u_next)
            max_p, argmax_p = torch.max(p_next, dim=-1)
            new_locks = (max_p >= confidence_lock_threshold) | (tau_next.squeeze(-1) >= 0.99)
            locked_mask = locked_mask | new_locks
            locked_tokens = torch.where(new_locks, argmax_p, locked_tokens)
            if locked_mask.any():
                one_hot = F.one_hot(locked_tokens, num_classes=vocab_size).float()
                u_next = torch.where(locked_mask.unsqueeze(-1), one_hot, u_next)
            u_curr = u_next
    probs = inverse_bhattacharyya_sphere_map(u_curr)
    tokens = torch.argmax(probs, dim=-1)
    return tokens, probs

print("Model architecture and 8-step Heun ODE solver compiled.")

In [ ]:
# Step 4: Fast GPU Training Loop (500 Steps)
seq_len, vocab_size = 64, 128
model = SimplexVelocityField(vocab_size=vocab_size, d_model=128, num_layers=4, num_heads=4, max_seq_len=seq_len).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Structured sequence data (prefix + modular rule)
data = torch.zeros(2000, seq_len, dtype=torch.long, device=device)
data[:, :seq_len//2] = torch.randint(1, vocab_size//2, (2000, seq_len//2), device=device)
data[:, seq_len//2:] = (data[:, :seq_len//2] * 3 + 7) % vocab_size
u1_all = bhattacharyya_sphere_map(F.one_hot(data, num_classes=vocab_size).float())
u0_uniform = torch.ones(vocab_size, device=device) / math.sqrt(vocab_size)

print("Training CWS-FM on GPU...")
t_start = time.time()
model.train()
for step in range(500):
    idx = torch.randint(0, 2000, (32,))
    u1 = u1_all[idx]
    t = torch.rand(32, 1, device=device)
    tau = wavefront_phase(t, seq_len, alpha=0.50).unsqueeze(-1)
    u0 = u0_uniform.view(1, 1, -1).expand(32, seq_len, -1)
    u_tau = geodesic_sphere_path(u0, u1, tau)
    v_target = geodesic_sphere_velocity(u0, u1, tau)
    v_pred = model(u_tau, t)
    loss = F.mse_loss(v_pred, v_target)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (step + 1) % 100 == 0 or step == 0:
        print(f"  Step {step+1:03d}/500 | Flow MSE Loss: {loss.item():.5f}")

print(f"Training complete in {time.time() - t_start:.2f}s.")

In [ ]:
# Step 5: GPU Inference Benchmark & Speed Verification
print("Benchmarking 8-Step Heun ODE Sampling on GPU:")
times = []
for _ in range(25):
    t0 = time.perf_counter()
    tokens, probs = sample_cws_fm(model, seq_len=seq_len, vocab_size=vocab_size, num_steps=8, device=device)
    times.append(time.perf_counter() - t0)

avg_ms = (sum(times) / len(times)) * 1000.0
tok_s = seq_len / (sum(times) / len(times))
max_p, _ = torch.max(probs, dim=-1)
collapse_rate = (max_p < 0.85).float().mean().item()

print(f"Latency: {avg_ms:.2f} ms | Throughput: {tok_s:.1f} tok/s | Rounding Collapse: {collapse_rate * 100:.2f}%")
print(f"Pre-Registered Gate 1 (Zero Rounding Collapse <= 5%): {'PASS' if collapse_rate <= 0.05 else 'FAIL'}")

In [ ]:
# Step 5b: 1500-Step Riemannian Simplex Flow Matching Benchmark
!python scripts/run_smoke.py --steps 1500 --batch-size 32 --seq-len 64


In [ ]:
# Step Pre-Kill: Explicit W&B Telemetry Cloud Upload
# Ensures every run (online or offline) is completely synced and uploaded to W&B cloud before teardown
!wandb sync --include-offline $(find /tmp/wandb ./wandb -type d -name "offline-run-*" 2>/dev/null) 2>/dev/null || true
!wandb sync --include-offline /tmp/wandb wandb 2>/dev/null || true


In [ ]:
# Step Final: Credit Protection & Machine Teardown
# @title Auto-Kill Runtime on Completion to Save Colab Credits
# Set to False by default so the virtual machine stays alive after execution
AUTO_KILL_ON_FINISH = False # @param {type:"boolean"}

if AUTO_KILL_ON_FINISH:
    print("\n[W&B Guard] Flushing all local offline runs to cloud before VM teardown...")
    try:
        sync_all_wandb_runs()
    except Exception as e:
        print(f"(W&B pre-teardown sync notice: {e})")

    print("\n[Credit Guard] Run complete. Unassigning Colab GPU immediately to prevent idle credit burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"(Auto-kill skipped: not running inside interactive Colab: {e})")
else:
    print("\n[Credit Guard] AUTO_KILL_ON_FINISH is False. Keeping VM instance alive for inspection.")
